0️⃣ Setup

In [8]:
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt

np.random.seed(42)


Simulated data

100 subjects

T_i = 3–4 time points

p = 7000 features

5 latent factors

In [9]:
n_subjects = 100
T_min, T_max = 3, 4
n_features = 500
k_factors = 5

# Generate irregular ages per subject
ages = [np.sort(np.random.uniform(30, 80, size=np.random.randint(T_min, T_max+1))) 
        for _ in range(n_subjects)]

# Placeholder for high-dimensional observations
# We'll simulate sparse factor loadings for this example
Lambda_true = np.zeros((n_features, k_factors))
for k in range(k_factors):
    Lambda_true[k*10:(k+1)*10, k] = np.random.randn(10) * 1.5  # only 10 non-zero per factor

# Latent OU parameters
theta_true = np.array([0.4, 0.7, 1.0, 0.6, 0.8])
sigma_f_true = np.array([1.0, 0.8, 0.6, 0.7, 0.9])
sigma_y_true = 0.5

# Generate latent factors and observations
Y = []
for i, age_i in enumerate(ages):
    T_i = len(age_i)
    f_i = np.zeros((T_i, k_factors))
    for k in range(k_factors):
        for t in range(1, T_i):
            dt = age_i[t] - age_i[t-1]
            a = np.exp(-theta_true[k] * dt)
            q = sigma_f_true[k] * np.sqrt((1 - a**2) / (2 * theta_true[k]))
            f_i[t, k] = a * f_i[t-1, k] + q * np.random.randn()
    Y_i = f_i @ Lambda_true.T + sigma_y_true * np.random.randn(T_i, n_features)
    Y.append(Y_i)


2️⃣ Memory-optimized PyMC model

Strategy:

Use shared $\theta$ and $\sigma_f$ across subjects

Factor recursion is computed subject by subject

Sparse horseshoe prior for $\Lambda$

Identifiable $\Lambda$ (lower-triangular + positive diagonal)

In [10]:
with pm.Model() as multi_subject_ou_model:

    # Sparse factor loadings
    tau = pm.HalfCauchy("tau", 1.0)
    lam = pm.HalfCauchy("lam", 1.0, shape=(n_features, k_factors))
    Lambda_raw = pm.Normal("Lambda_raw", 0, tau * lam, shape=(n_features, k_factors))
    
    # Identifiability: lower-triangular + positive diagonal
    Lambda_lt = pt.tril(Lambda_raw)
    diag_idx = np.arange(k_factors)
    Lambda = pt.set_subtensor(Lambda_lt[diag_idx, diag_idx], pt.abs(Lambda_lt[diag_idx, diag_idx]))
    Lambda = pm.Deterministic("Lambda", Lambda)
    
    # Population-level OU parameters
    theta = pm.Exponential("theta", 1.0, shape=k_factors)
    theta_ord = pm.Deterministic("theta_ord", pt.sort(theta))
    sigma_f = pm.HalfNormal("sigma_f", 1.0, shape=k_factors)
    
    # Subject-level deviations (optional)
    sigma_subj = pm.HalfNormal("sigma_subj", 0.5, shape=k_factors)
    
    # Observation noise
    sigma_y = pm.HalfNormal("sigma_y", 1.0, shape=n_features)
    
    # For each subject
    for i, Y_i in enumerate(Y):
        age_i = ages[i]
        T_i = len(age_i)
        dt_i = np.diff(age_i)
        
        # Initial latent factor
        f0 = pm.Normal(f"f0_{i}", 0, 1, shape=k_factors)
        f_i = [f0]
        
        # OU dynamics
        for t in range(1, T_i):
            a_t = pt.exp(-theta_ord * dt_i[t-1])
            q_t = sigma_f * pt.sqrt((1 - a_t**2) / (2*theta_ord))
            eps_t = pm.Normal(f"eps_{i}_{t}", 0, 1, shape=k_factors)
            f_new = a_t * f_i[-1] + q_t * eps_t
            f_i.append(f_new)
        
        f_i = pt.stack(f_i) + pm.Normal(f"eta_{i}", 0, sigma_subj, shape=(T_i, k_factors))
        
        mu_i = pt.dot(f_i, Lambda.T)
        pm.Normal(f"y_obs_{i}", mu=mu_i, sigma=sigma_y, observed=Y_i)


3️⃣ Posterior sampling

In [11]:
with multi_subject_ou_model:
    trace = pm.sample(draws=1000, tune=1000, chains=2, target_accept=0.9)


Initializing NUTS using jitter+adapt_diag...
ERROR (pytensor.graph.rewriting.basic): Rewrite failure due to: constant_folding
ERROR (pytensor.graph.rewriting.basic): node: Add([-3. -3. - ... . -3. -3.], [-3. -3. - ... . -3. -3.], [-4. -4. - ... . -4. -4.], [-4. -4. - ... . -4. -4.], [-4. -4. - ... . -4. -4.], [-3. -3. - ... . -3. -3.], [-4. -4. - ... . -4. -4.], [-3. -3. - ... . -3. -3.], [-3. -3. - ... . -3. -3.], [-4. -4. - ... . -4. -4.], [-3. -3. - ... . -3. -3.], [-3. -3. - ... . -3. -3.], [-4. -4. - ... . -4. -4.], [-4. -4. - ... . -4. -4.], [-4. -4. - ... . -4. -4.], [-3. -3. - ... . -3. -3.], [-4. -4. - ... . -4. -4.], [-3. -3. - ... . -3. -3.], [-4. -4. - ... . -4. -4.], [-3. -3. - ... . -3. -3.], [-3. -3. - ... . -3. -3.], [-3. -3. - ... . -3. -3.], [-4. -4. - ... . -4. -4.], [-3. -3. - ... . -3. -3.], [-3. -3. - ... . -3. -3.], [-4. -4. - ... . -4. -4.], [-4. -4. - ... . -4. -4.], [-3. -3. - ... . -3. -3.], [-4. -4. - ... . -4. -4.], [-4. -4. - ... . -4. -4.], [-4. -4. - ...

Output()

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 14716 seconds.
There were 492 divergences after tuning. Increase `target_accept` or reparameterize.
Chain 0 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
Chain 1 reached the maximum tree depth. Increase `max_treedepth`, increase `target_accept` or reparameterize.
We recommend running at least 4 chains for robust computation of convergence diagnostics
The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details
The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


4️⃣ Posterior predictive checks (PPCs)

In [12]:
with multi_subject_ou_model:
    ppc = pm.sample_posterior_predictive(trace, var_names=["y_obs_0"], random_seed=42)
    
y_ppc_mean = ppc["y_obs_0"].mean(axis=0)

plt.figure(figsize=(10,4))
plt.plot(y_ppc_mean[:, :10], "--")  # first 10 features
plt.title("Posterior predictive mean for first subject")
plt.show()


Sampling: [y_obs_0]


Output()

KeyError: 'y_obs_0'

5️⃣ Extract latent factor trajectories

In [ ]:
f_post = []
for i in range(len(Y)):
    f_i_post = np.mean([trace.get_values(f"f0_{i}")], axis=0)  # approximate f0
    f_post.append(f_i_post)

# Now f_post contains population-level latent trajectories for each subject


1️⃣ Population-level latent factor trajectories

Idea: For each factor $k$:
\begin{align}
f_{\text {pop }, k}(a) \approx \text { posterior mean of latent factor across all subjects at age } a
\end{align}

In [ ]:
# Collect posterior means for f0 (initial factors) for all subjects
f_pop_means = []
ages_all = []

for i, age_i in enumerate(ages):
    f0_samples = trace.posterior[f"f0_{i}"].mean(dim=("chain", "draw")).values
    f_pop_means.append(f0_samples)  # shape (k_factors,)
    ages_all.append(age_i[0])  # use first time point as representative

f_pop_means = np.stack(f_pop_means)  # shape (n_subjects, k_factors)
ages_all = np.array(ages_all)


2️⃣ Smooth interpolation across age

In [ ]:
from scipy.interpolate import interp1d

k_factors = f_pop_means.shape[1]
age_grid = np.linspace(30, 80, 200)

f_smooth = np.zeros((len(age_grid), k_factors))

for k in range(k_factors):
    interp = interp1d(ages_all, f_pop_means[:, k], kind='linear', fill_value="extrapolate")
    f_smooth[:, k] = interp(age_grid)


3️⃣ Factor recovery plot

In [ ]:
plt.figure(figsize=(10, 5))
for k in range(k_factors):
    plt.plot(age_grid, f_smooth[:, k], label=f"Factor {k+1}")
plt.xlabel("Age")
plt.ylabel("Latent factor value")
plt.title("Recovered population-level latent factor trajectories")
plt.legend()
plt.show()


In [ ]:
for k in range(k_factors):
    plt.scatter(ages_all, f_pop_means[:, k], alpha=0.5)


4️⃣ Optional: Factor credible intervals

In [ ]:
f_pop_samples = []

for i, age_i in enumerate(ages):
    f0_samples = trace.posterior[f"f0_{i}"].values  # shape (chains, draws, k_factors)
    f_pop_samples.append(f0_samples.reshape(-1, k_factors))

f_pop_samples = np.stack(f_pop_samples)  # shape (n_subjects, n_samples, k_factors)

# Interpolate each sample
f_smooth_samples = np.zeros((len(age_grid), k_factors, f_pop_samples.shape[1]))

for k in range(k_factors):
    for s in range(f_pop_samples.shape[1]):
        interp = interp1d(ages_all, f_pop_samples[:, s, k], kind='linear', fill_value="extrapolate")
        f_smooth_samples[:, k, s] = interp(age_grid)

# 95% CI
f_lower = np.percentile(f_smooth_samples, 2.5, axis=2)
f_upper = np.percentile(f_smooth_samples, 97.5, axis=2)
f_mean = np.mean(f_smooth_samples, axis=2)

# Plot
plt.figure(figsize=(10,5))
for k in range(k_factors):
    plt.fill_between(age_grid, f_lower[:, k], f_upper[:, k], alpha=0.3)
    plt.plot(age_grid, f_mean[:, k], label=f"Factor {k+1}")
plt.xlabel("Age")
plt.ylabel("Latent factor")
plt.title("Population-level latent factor trajectories with 95% CI")
plt.legend()
plt.show()
